# Advanced Problems with Solutions: Decorators and Memoization

Topics: recursive memoization, generic decorators, `*args` / `**kwargs`, hashability issues, cache inspection, cache invalidation, `functools.lru_cache`, and decorator stacking.

## Problem 1 — Build a Recursive Memoization Decorator

Write a decorator `memoize` that caches results of a recursive Fibonacci function.

Requirements:

- preserve function metadata with `functools.wraps`
- store results in a dictionary
- expose the cache as `.cache`
- prove that repeated calls reuse cached values

In [1]:
from functools import wraps

def memoize(fn):
    cache = {}

    @wraps(fn)
    def inner(*args):
        if args not in cache:
            cache[args] = fn(*args)
        return cache[args]

    inner.cache = cache
    return inner

@memoize
def fib(n):
    """Return the nth Fibonacci number using 1, 1 as the base sequence."""
    print(f"Calculating fib({n})")
    return 1 if n < 3 else fib(n - 1) + fib(n - 2)

assert fib(6) == 8
assert fib(6) == 8
assert fib.__name__ == "fib"
assert fib.__doc__.startswith("Return the nth Fibonacci")
assert (6,) in fib.cache

fib.cache

Calculating fib(6)
Calculating fib(5)
Calculating fib(4)
Calculating fib(3)
Calculating fib(2)
Calculating fib(1)


{(2,): 1, (1,): 1, (3,): 2, (4,): 3, (5,): 5, (6,): 8}

## Problem 2 — Support Keyword Arguments Safely

Improve `memoize` so it supports both positional arguments and keyword arguments.

The following calls should hit the same cache entry:

```python
combine(2, b=3)
combine(2, b=3)
```

But different keyword values should create different cache entries.

In [2]:
from functools import wraps

def memoize_kwargs(fn):
    cache = {}

    @wraps(fn)
    def inner(*args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        if key not in cache:
            cache[key] = fn(*args, **kwargs)
        return cache[key]

    inner.cache = cache
    return inner

calls = {"count": 0}

@memoize_kwargs
def combine(a, b=0, scale=1):
    calls["count"] += 1
    return (a + b) * scale

assert combine(2, b=3) == 5
assert combine(2, b=3) == 5
assert combine(2, b=4) == 6
assert combine(2, b=3, scale=10) == 50
assert calls["count"] == 3

combine.cache

{((2,), (('b', 3),)): 5,
 ((2,), (('b', 4),)): 6,
 ((2,), (('b', 3), ('scale', 10))): 50}

## Problem 3 — Handle Unhashable Arguments Gracefully

A normal dictionary-based cache fails when arguments contain lists or dictionaries.

Write a `freeze` helper that converts common mutable containers into hashable equivalents.

Then use it in a memoization decorator.

In [3]:
from functools import wraps

def freeze(value):
    if isinstance(value, dict):
        return tuple(sorted((freeze(k), freeze(v)) for k, v in value.items()))
    if isinstance(value, (list, tuple)):
        return tuple(freeze(item) for item in value)
    if isinstance(value, set):
        return tuple(sorted(freeze(item) for item in value))
    return value

def robust_memoize(fn):
    cache = {}

    @wraps(fn)
    def inner(*args, **kwargs):
        key = (freeze(args), freeze(kwargs))
        if key not in cache:
            cache[key] = fn(*args, **kwargs)
        return cache[key]

    inner.cache = cache
    return inner

calls = {"count": 0}

@robust_memoize
def total_nested(data):
    calls["count"] += 1
    return sum(data)

assert total_nested([1, 2, 3]) == 6
assert total_nested([1, 2, 3]) == 6
assert calls["count"] == 1

total_nested.cache

{(((1, 2, 3),), ()): 6}

## Problem 4 — Add Cache Statistics and Cache Clearing

Create a memoization decorator that tracks:

- cache hits
- cache misses
- current cache size

Also expose a `.cache_clear()` method.

In [4]:
from functools import wraps

def memoize_with_stats(fn):
    cache = {}
    stats = {"hits": 0, "misses": 0}

    @wraps(fn)
    def inner(*args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        if key in cache:
            stats["hits"] += 1
            return cache[key]

        stats["misses"] += 1
        cache[key] = fn(*args, **kwargs)
        return cache[key]

    def cache_info():
        return {
            "hits": stats["hits"],
            "misses": stats["misses"],
            "size": len(cache)
        }

    def cache_clear():
        cache.clear()
        stats["hits"] = 0
        stats["misses"] = 0

    inner.cache = cache
    inner.cache_info = cache_info
    inner.cache_clear = cache_clear
    return inner

@memoize_with_stats
def slow_add(a, b):
    return a + b

assert slow_add(1, 2) == 3
assert slow_add(1, 2) == 3
assert slow_add(2, 3) == 5

assert slow_add.cache_info() == {"hits": 1, "misses": 2, "size": 2}

slow_add.cache_clear()
assert slow_add.cache_info() == {"hits": 0, "misses": 0, "size": 0}

slow_add.cache_info()

{'hits': 0, 'misses': 0, 'size': 0}

## Problem 5 — Compare Manual Memoization with `lru_cache`

Use `functools.lru_cache` to memoize Fibonacci.

Then inspect cache statistics with `.cache_info()`.

In [5]:
from functools import lru_cache

@lru_cache(maxsize=8)
def fib_lru(n):
    return 1 if n < 3 else fib_lru(n - 1) + fib_lru(n - 2)

assert fib_lru(10) == 55
first_info = fib_lru.cache_info()

assert fib_lru(10) == 55
second_info = fib_lru.cache_info()

assert second_info.hits > first_info.hits

print(first_info)
print(second_info)

CacheInfo(hits=7, misses=10, maxsize=8, currsize=8)
CacheInfo(hits=8, misses=10, maxsize=8, currsize=8)


## Problem 6 — Demonstrate LRU Eviction

Use `@lru_cache(maxsize=3)` and prove that older entries are evicted when the cache exceeds capacity.

In [6]:
from functools import lru_cache

calls = {"count": 0}

@lru_cache(maxsize=3)
def identity(x):
    calls["count"] += 1
    print(f"Computing identity({x})")
    return x

identity(1)
identity(2)
identity(3)
identity(1)
identity(4)
identity(2)

info = identity.cache_info()
print(info)

assert info.maxsize == 3
assert info.currsize == 3
assert calls["count"] == 5

Computing identity(1)
Computing identity(2)
Computing identity(3)
Computing identity(4)
Computing identity(2)
CacheInfo(hits=1, misses=5, maxsize=3, currsize=3)


## Problem 7 — Decorator Stacking with Memoization and Logging

Show the difference between:

```python
@logged
@memoize
```

and:

```python
@memoize
@logged
```

Explain why one logs every call while the other logs only cache misses.

In [7]:
from functools import wraps

def logged(fn):
    @wraps(fn)
    def inner(*args, **kwargs):
        print(f"LOG: calling {fn.__name__}{args}{kwargs}")
        return fn(*args, **kwargs)
    return inner

def simple_memoize(fn):
    cache = {}

    @wraps(fn)
    def inner(*args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        if key not in cache:
            cache[key] = fn(*args, **kwargs)
        return cache[key]

    return inner

@logged
@simple_memoize
def square_a(n):
    print("Computing square_a")
    return n * n

@simple_memoize
@logged
def square_b(n):
    print("Computing square_b")
    return n * n

print("square_a:")
square_a(5)
square_a(5)

print("\nsquare_b:")
square_b(5)
square_b(5)

square_a:
LOG: calling square_a(5,){}
Computing square_a
LOG: calling square_a(5,){}

square_b:
LOG: calling square_b(5,){}
Computing square_b


25

Solution explanation:

`square_a = logged(simple_memoize(square_a))`, so logging happens before cache lookup on every call.

`square_b = simple_memoize(logged(square_b))`, so the cache wrapper runs first. If the result is cached, the logged function is never called.

## Problem 8 — Time-to-Live Cache Decorator

Create a decorator factory `ttl_cache(seconds)`.

A cached value should be reused only while it is younger than the configured TTL.

In [8]:
from functools import wraps
from time import monotonic, sleep

def ttl_cache(seconds):
    def decorator(fn):
        cache = {}

        @wraps(fn)
        def inner(*args, **kwargs):
            key = (args, tuple(sorted(kwargs.items())))
            now = monotonic()

            if key in cache:
                created_at, value = cache[key]
                if now - created_at < seconds:
                    return value

            value = fn(*args, **kwargs)
            cache[key] = (now, value)
            return value

        inner.cache = cache
        return inner
    return decorator

calls = {"count": 0}

@ttl_cache(seconds=0.2)
def expensive_value(x):
    calls["count"] += 1
    return x * 10

assert expensive_value(3) == 30
assert expensive_value(3) == 30
assert calls["count"] == 1

sleep(0.25)
assert expensive_value(3) == 30
assert calls["count"] == 2

calls

{'count': 2}